In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from keras import regularizers, layers

In [ ]:
tf.config.list_physical_devices('GPU')

In [ ]:
train_ds, val_ds = tf.keras.utils.image_dataset_from_directory(
  '/kaggle/input/mnist-dcgan-output-35/generated_images_with_images',
  validation_split=0.2,
  subset="both",
  seed=123,
  image_size=(28, 28),
  batch_size=32, color_mode='grayscale')

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
model = keras.Sequential([
  layers.Rescaling(1./255, input_shape=(28, 28, 1)),
  layers.Conv2D(64, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.BatchNormalization(),
  layers.Dropout(.3),
  layers.Conv2D(128, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.BatchNormalization(),
  layers.Dropout(.3),
  layers.Conv2D(64, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.BatchNormalization(),
  layers.Dropout(.3),
  layers.Flatten(),
  layers.Dense(28, activation='relu'),
  layers.Dense(10, activation='softmax')
])

In [ ]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
              metrics=['accuracy'])

In [ ]:
epochs=50
history = model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=epochs
)

In [ ]:
test_ds = tf.keras.utils.image_dataset_from_directory(
  '/kaggle/input/mnist-png/mnist_png/test',
  seed=123,
  image_size=(28, 28),
  batch_size=32, color_mode='grayscale')

In [ ]:
model.evaluate(test_ds)